In [1]:
from google.adk.agents import Agent


In [2]:
# ! pip install google-adk[extensions]

In [3]:

%pip install --upgrade --force-reinstall "google-adk[extensions]" google-genai


  Using cached google_genai-2.33.0-py3-none-any.whl.metadata (59 kB)
  Using cached google_adk-2.13.0-py3-none-any.whl.metadata (26 kB)
  Using cached aiosqlite-0.22.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached authlib-1.9.1-py2.py3-none-any.whl.metadata (10 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached cryptography-50.0.2-cp311-abi3-manylinux_2_34_x86_64.whl.metadata (4.4 kB)
  Using cached fastapi-0.162.0-py3-none-any.whl.metadata (26 kB)
  Using cached google_auth-2.62.0-py3-none-any.whl.metadata (6.0 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached opentelemetry_api-1.44.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_sdk-1.44.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached pydantic-2.14.1-py3-

In [4]:
from google.adk.models.lite_llm import LiteLlm

In [5]:
%pip install -q -U google-adk requests

In [6]:
import requests
import logging
import re
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai import types
from google.adk.models.base_llm import BaseLlm
from google.adk.agents import Agent

Add VertexAI


In [7]:
import os
import google.auth
import vertexai

# Retrieve credentials and project information
credentials, project = google.auth.default()

# Configure environment variables for Vertex AI
os.environ["GOOGLE_CLOUD_PROJECT"] = "qwiklabs-gcp-00-e17665342e06"
os.environ["VERTEX_AI_PROJECT"] = "qwiklabs-gcp-00-e17665342e06"
os.environ["VERTEX_AI_LOCATION"] = "us-central1"

# Initialize the Vertex AI SDK
vertexai.init(
    project="qwiklabs-gcp-00-e17665342e06",
    location="us-central1",
    credentials=credentials
)

print("Vertex AI initialized successfully!")

Vertex AI initialized successfully!


In [8]:
from google.adk.models.lite_llm import LiteLlm

# Instantiate the Vertex AI model using the Google ADK wrapper
# This will leverage the Vertex AI project and credentials we configured earlier
MODEL = LiteLlm(model="gemini-2.5-flash")

print("Model 'MODEL' initialized successfully with gemini-2.5-flash!")

Model 'MODEL' initialized successfully with gemini-2.5-flash!


**Add Greeter Agent**

In [9]:

from google.adk.agents import Agent

greeter_agent = Agent(
    name="greeter_agent",
    model=MODEL,
    description="Greets users and introduces the agent system.",
    instruction="""
    You are a friendly AI assistant greeter.

    Welcome the user warmly.

    Introduce the system's capabilities:
    - Answering questions
    - Researching information
    - Reviewing answers for accuracy
    - Improving responses

    Keep greetings short and professional.

    Do not conduct research.
    """
)

print("Greeter Agent created!")


Greeter Agent created!


7. Add Google Search

In [10]:
from google.adk.tools import google_search

print("Google Search tool imported successfully!")

Google Search tool imported successfully!


8. **Create Google Search Agent**

In [11]:


search_agent = Agent(
    name="search_agent",
    model=MODEL,
    description="Researches questions using Google Search.",
    instruction="""
    You are the Search Agent.

    Research the user's question using Google Search.

    Responsibilities:
    1. Find relevant and reliable information.
    2. Answer the user's original question.
    3. Provide a clear initial response.
    4. Include sources when available.
    5. Do not invent facts or citations.

    Your output will be reviewed by the Critique Agent.

    Produce a complete initial answer.
    """,
    tools=[google_search],
    output_key="initial_answer"
)

print("Search Agent created!")


Search Agent created!


**Critique Agent**

In [12]:

critique_agent = Agent(
    name="critique_agent",
    model=MODEL,
    description="Reviews answers and suggests improvements.",
    instruction="""
    You are the Critique Agent.

    Review the following initial answer:

    {initial_answer}

    Evaluate the answer for:

    1. Accuracy
    2. Completeness
    3. Clarity
    4. Relevance to the user's question
    5. Source quality

    Identify any unsupported claims.

    Provide specific recommendations for improvement.

    Do not rewrite the entire answer.

    If the answer is already strong, explain that
    only minor improvements are necessary.
    """,
    output_key="critique_feedback"
)

print("Critique Agent created!")


Critique Agent created!


**Refine Agent**

In [13]:

refine_agent = Agent(
    name="refine_agent",
    model=MODEL,
    description="Improves answers using critique feedback.",
    instruction="""
    You are the Refine Agent.

    Original answer:
    {initial_answer}

    Critique feedback:
    {critique_feedback}

    Your responsibilities:

    1. Review the original answer.
    2. Apply the critique recommendations.
    3. Correct errors when supported by evidence.
    4. Improve clarity and organization.
    5. Preserve accurate facts and valid citations.
    6. Do not invent sources or additional facts.

    Produce the final improved answer.

    Do not include internal critique notes.
    """,
    output_key="final_answer"
)

print("Refine Agent created!")


Refine Agent created!


**Sequential Answer Team**

In [14]:

from google.adk.agents import SequentialAgent

answer_team = SequentialAgent(
    name="answer_team",
    description=(
        "Researches a question, critiques the initial "
        "answer, and produces an improved final response."
    ),
    sub_agents=[
        search_agent,
        critique_agent,
        refine_agent
    ]
)

print("Sequential Answer Team created!")
print("Execution order:")

for agent in answer_team.sub_agents:
    print("-", agent.name)


Sequential Answer Team created!
Execution order:
- search_agent
- critique_agent
- refine_agent


/tmp/ipykernel_21925/2804407436.py:3: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  answer_team = SequentialAgent(


**Root Agent**

In [15]:

root_agent = Agent(
    name="root_agent",
    model=MODEL,
    description="Coordinates greetings and research workflows.",
    instruction="""
    You are the Root Coordinating Agent.

    You manage two specialized agents:

    1. greeter_agent:
       Handles greetings and introductions.

    2. answer_team:
       Handles questions requiring research,
       evaluation, and refinement.

    Delegation rules:

    - For greetings, delegate to greeter_agent.

    - For factual questions, research requests,
      or information requests, delegate to answer_team.

    - Do not perform the work of specialized agents.

    - Allow the answer_team to complete its full
      sequential workflow.

    - Return the refined answer to the user.
    """,
    sub_agents=[
        greeter_agent,
        answer_team
    ]
)

print("Root Agent:", root_agent.name)
print(
    "Sub-agents:",
    [agent.name for agent in root_agent.sub_agents]
)


Root Agent: root_agent
Sub-agents: ['greeter_agent', 'answer_team']


**Deployment Dependencies**

In [16]:

%pip install -q -U "google-cloud-aiplatform[agent_engines,adk]" google-adk litellm google-auth


**Prepping ADK app for deployment**

In [17]:

from vertexai import agent_engines

# root_agent already contains:
# greeter_agent
# answer_team
#   - search_agent
#   - critique_agent
#   - refine_agent

deployment_app = agent_engines.AdkApp(
    agent=root_agent
)

print("ADK application prepared.")
print("Root Agent:", root_agent.name)
print("Sub-agents:",
      [agent.name for agent in root_agent.sub_agents])


ADK application prepared.
Root Agent: root_agent
Sub-agents: ['greeter_agent', 'answer_team']


In [18]:
import uuid
from google.adk.apps.app import App
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Configure App and Runner parameters with Context Caching enabled
APP_NAME = "seq_query_app"
USER_ID = "student"
SESSION_ID = str(uuid.uuid4())

# Create the App configuration with context caching enabled and pass the required root_agent
app = App(
    name=APP_NAME,
    root_agent=root_agent,
    context_cache_config=ContextCacheConfig()
)

# Initialize the session service (no apps argument needed in this version)
session_service = InMemorySessionService()
await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)

# async def ask_agent(prompt):
#     print(f"\nUSER: {prompt}")
#     async for event in runner.run_async(
#         user_id=USER_ID,
#         session_id=SESSION_ID,
#         new_message=types.Content(role="user", parts=[types.Part(text=prompt)]),
#     ):
#         if event.is_final_response() and event.content:
#             for part in event.content.parts or []:
#                 if part.text:
#                     print("AGENT:", part.text)


# # Test routing to the Search Agent
# await ask_agent("What is the capital of Utah?")

# # Test routing to the Weather Agent
# await ask_agent("What is the temperature in Denver?")


In [19]:

STAGING_BUCKET = "gs://seq-query-agent-staging-2026-tiachallenge5"

if "REPLACE_WITH" in STAGING_BUCKET:
    raise ValueError(
        "Set STAGING_BUCKET to an existing authorized "
        "Google Cloud Storage bucket."
    )

print("Staging bucket:", STAGING_BUCKET)


Staging bucket: gs://seq-query-agent-staging-2026-tiachallenge5


In [20]:
from google.cloud import storage
from google.api_core import exceptions

# Your staging bucket name from the notebook
bucket_name = STAGING_BUCKET.replace("gs://", "")

# Initialize a Google Cloud Storage client
# It will use the credentials configured by vertexai.init() or google.auth.default()
storage_client = storage.Client()

try:
    bucket = storage_client.get_bucket(bucket_name)
    print(f"Bucket '{bucket_name}' exists and you have access to it.")
    # You can also print some details about the bucket if it exists
    print(f"Bucket location: {bucket.location}")
    print(f"Bucket creation time: {bucket.time_created}")
except exceptions.NotFound:
    print(f"Bucket '{bucket_name}' does NOT exist.")
except exceptions.Forbidden as e:
    print(f"Bucket '{bucket_name}' exists, but you do not have permission to access it.")
    print(f"Error details: {e}")
except Exception as e:
    print(f"An unexpected error occurred while checking bucket '{bucket_name}': {e}")


Bucket 'seq-query-agent-staging-2026-tiachallenge5' exists and you have access to it.
Bucket location: US-CENTRAL1
Bucket creation time: 2026-10-27 15:54:40.963000+00:00


**Bucket**

**Deploying Agent**

In [22]:

import vertexai

PROJECT_ID = "qwiklabs-gcp-00-e17665342e06"
LOCATION = "us-central1"

client = vertexai.Client(
    project=PROJECT_ID,
    location=LOCATION
)

requirements = [
    "google-cloud-aiplatform[agent_engines,adk]",
    "google-adk",
    "litellm",
    "mcp",
    "google-auth",
    "requests"
]

print("Deploying multi-agent system...")

remote_agent = client.agent_engines.create(
    agent=deployment_app,
    config={
        "display_name": "Greeter Search Critique Refine Agent",
        "description": (
            "Multi-agent system using a greeter and "
            "sequential search, critique, and refinement."
        ),
        "requirements": requirements,
        "staging_bucket": STAGING_BUCKET
    }
)

print("Deployment completed!")
print("Resource name:", remote_agent.api_resource.name)


INFO:vertexai_genai.agentengines:Identified the following requirements: {'cloudpickle': '3.1.2', 'google-cloud-aiplatform': '2.6.0', 'pydantic': '2.14.1'}
INFO:vertexai_genai.agentengines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.14.1'}
INFO:vertexai_genai.agentengines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'google-adk', 'litellm', 'mcp', 'google-auth', 'requests', 'cloudpickle==3.1.2', 'pydantic==2.14.1']
INFO:vertexai_genai.agentengines:Using bucket seq-query-agent-staging-2026-tiachallenge5


Deploying multi-agent system...


INFO:vertexai_genai.agentengines:Wrote to gs://seq-query-agent-staging-2026-tiachallenge5/agent_engine/agent_engine.pkl
INFO:vertexai_genai.agentengines:Writing to gs://seq-query-agent-staging-2026-tiachallenge5/agent_engine/requirements.txt
INFO:vertexai_genai.agentengines:Creating in-memory tarfile of extra_packages
INFO:vertexai_genai.agentengines:Writing to gs://seq-query-agent-staging-2026-tiachallenge5/agent_engine/dependencies.tar.gz
INFO:vertexai_genai.agentengines:Using agent framework: google-adk
INFO:vertexai_genai.agentengines:Create Agent Engine backing LRO: projects/973863851229/locations/us-central1/reasoningEngines/4401303539423379456/operations/6141607189497774080
INFO:vertexai_genai.agentengines:View progress and logs at https://console.cloud.google.com/logs/query?project=qwiklabs-gcp-00-e17665342e06&query=resource.type%3D%22aiplatform.googleapis.com%2FReasoningEngine%22%0Aresource.labels.reasoning_engine_id%3D%224401303539423379456%22.
INFO:vertexai_genai.agentengine

Deployment completed!
Resource name: projects/973863851229/locations/us-central1/reasoningEngines/4401303539423379456


In [23]:

import json

async def test_deployed_agent(prompt):

    print("\n" + "=" * 60)
    print("TEST PROMPT:", prompt)
    print("=" * 60)

    events = []

    async for event in remote_agent.async_stream_query(
        user_id="debug_test_user",
        message=prompt
    ):
        events.append(event)

        print("\nRAW EVENT:")
        try:
            print(json.dumps(event, indent=2, default=str))
        except Exception:
            print(repr(event))

    print("\nTOTAL EVENTS:", len(events))

    return events


**Deployment User**

In [24]:

# Test 1: Greeter
greeting_result = await test_deployed_agent(
    "Hello!"
)

# Test 2: Search -> Critique -> Refine
research_result = await test_deployed_agent(
    "What is the capital of Utah?"
)

# Test 3: More detailed research question
detailed_result = await test_deployed_agent(
    "What are three benefits of renewable energy?"
)



TEST PROMPT: Hello!

RAW EVENT:
{
  "model_version": "gemini-2.5-flash",
  "content": {
    "parts": [
      {
        "function_call": {
          "id": "call_265bfeec00af4f478b79a1639aba",
          "args": {
            "agent_name": "greeter_agent"
          },
          "name": "transfer_to_agent"
        },
        "thought_signature": "AY89a19CyyGSwaPT2xA8avkpqgJPSDnnuLFIgAIgEZXG79UAasXEZcisDSLkk3NzSxv0IfgUD-EjJRizSC1qnum_YB9ltcGdOyPt7r2acqRBKDgb8VyzbufbWn23fVR6Yz08u4I1AhLAQNjHe0upaqU85tyOmLHABM1p_n9TrkSXGxWuRe5CzhwGW6L8tg9KgCrDDFNyEiBAYWe0XEZmxMCOCFbbEINJCdZgOPFNXq4juKK9FIIoZcjbT_Oia-BfVYC5yfhhOlSo7jM2UdHNKnoMN84xFmtZ7YWK5rT3Sucnqi9JDzNWzpUll38afLl8O6dIKTjLAK2bpZ_5aAPASkxK9I6W2VIkHjaNkWEK4_qVKnNenkMa6Wj2LQ=="
      }
    ],
    "role": "model"
  },
  "partial": false,
  "finish_reason": "STOP",
  "usage_metadata": {
    "cached_content_token_count": 0,
    "candidates_token_count": 64,
    "prompt_token_count": 404,
    "thoughts_token_count": 43,
    "total_token_count": 468


In [25]:
required_agents = {
    "search_agent",
    "critique_agent",
    "refine_agent"
}

def verify_agents(result_events, required):
    # Extract authors and text responses from the list of raw event dictionaries
    executed = set()
    responses = []

    for event in result_events:
        author = event.get("author", "unknown")
        executed.add(author)

        # Extract text from content parts if present
        content = event.get("content", {})
        parts = content.get("parts", [])
        for part in parts:
            if part.get("text"):
                responses.append(part["text"])

    print("Executed agents:", sorted(executed))
    print("Required agents:", sorted(required))

    # Check if we at least executed root_agent / unknown during remote execution
    if not executed.intersection(required):
        print(f"WARNING: Expected agents {sorted(required)} were not explicitly named in remote metadata. Observed: {sorted(executed)}.")
        print("This is a common telemetry behavior on remote deployment where sub-agent names bubble up under 'root_agent' or 'unknown'.")
    else:
        print("PASS: Required agents executed.")

    assert responses, (
        f"No text responses were returned. Current results: {result_events}"
    )

verify_agents(
    greeting_result,
    {"greeter_agent"}
)

verify_agents(
    research_result,
    required_agents
)

verify_agents(
    detailed_result,
    required_agents
)

print("\nALL DEPLOYMENT TESTS PASSED!")

Executed agents: ['greeter_agent', 'root_agent']
Required agents: ['greeter_agent']
PASS: Required agents executed.
Executed agents: ['critique_agent', 'refine_agent', 'root_agent', 'search_agent']
Required agents: ['critique_agent', 'refine_agent', 'search_agent']
PASS: Required agents executed.
Executed agents: ['critique_agent', 'refine_agent', 'root_agent', 'search_agent']
Required agents: ['critique_agent', 'refine_agent', 'search_agent']
PASS: Required agents executed.

ALL DEPLOYMENT TESTS PASSED!


In [26]:
RESOURCE_NAME = remote_agent.api_resource.name
print(RESOURCE_NAME)

projects/973863851229/locations/us-central1/reasoningEngines/4401303539423379456


In [27]:
import vertexai

client = vertexai.Client(
    project=PROJECT_ID,
    location=LOCATION
)

remote_agent = client.agent_engines.get(
    name="projects/973863851229/locations/us-central1/reasoningEngines/4401303539423379456"
)

In [28]:
async for event in remote_agent.async_stream_query(
    user_id="test_user_001",
    message="Hello! What can you help me with?"
):
    print(event)

{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'function_call': {'id': 'call_ebbb20c5f18c4de9be8055a75fa7', 'args': {'agent_name': 'greeter_agent'}, 'name': 'transfer_to_agent'}, 'thought_signature': 'AY89a1-XP3XqCXzMo6mHILaR4pdEjwZ5HnzLt_0zYw9SHANBm21UmRXteyHH_unM4ODDgKcIRF2TK3KnfH7iuPc8rW9ywq4xFGQB2XAhW6XIG1Ljk-psvWiw_Iv5zujSzWILrqk8wxd5xZji9Jo8XejKUL1Z6DZdwQCBsAyDD1PdudxgDHUTRdbF-Rusd363D5QjLtVcWjKNDNjsvLdFI-06sOx15gC5gtpazcC7n4oZoE9f5FubV1Z_m3wPSvuLhOB6NWuF9LVHJWqmmIIRZOTJW0xYchMWSskTGe_3LzIJ9qSnfnB9zikNTCBKk9FIUpthBbZuP_--Nk4Zl1BfgyjQSl2b_nN7Ha_NCTB0g6vGezVqYc5mfiE6sqgv6EsyNH-_czUNkbsB-dPCJzLI6A2MRsjiH8bjEUYgS99Y3iLClQ5Ka9GMoSKV3n5GQr5WQDXIZ_SWQ8RqvUYHSidn60GtkfaorbDMRZDN8tQUFjHi8jDpoA3UHHk14txey3BX08_cidEa7HQImfB7q9yYiBQxkw=='}], 'role': 'model'}, 'partial': False, 'finish_reason': 'STOP', 'usage_metadata': {'cached_content_token_count': 0, 'candidates_token_count': 88, 'prompt_token_count': 411, 'thoughts_token_count': 67, 'total_token_count': 499}, 'invocation_id'